# GDP ETL Preview Notebook

Runs the same extract -> transform -> load steps as the `gdp_etl` Airflow DAG (`dags/gdp_etl_dag.py`), in-process instead of as three separate Kubernetes pods, and previews the resulting data.

It calls the exact same functions the DAG's pods run (`src/etl/extract.py`, `src/etl/transform.py`, `src/etl/load.py`) so there is no separate logic to keep in sync -- only the pod-to-pod file hand-off via the `mlpipeline-etl-data` PVC is skipped, since everything runs in one process here.

## Prerequisites

1. Activate the project venv and make sure dependencies are installed:
   ```bash
   source venv/bin/activate
   pip install -r requirements.txt
   pip install jupyter ipykernel  # not in requirements.txt; notebook-only tooling
   ```
2. Set your BEA API key (same value as in `kubernetes/etl-secret.yaml`'s `BEA_API_KEY` -- do not paste it into this notebook, only export it in your shell):
   ```bash
   export BEA_API_KEY=your-real-key
   ```
3. The **Preview existing data** section needs a connection to the live `mlpipeline-postgres` instance. It's `ClusterIP`-only, so port-forward it first, in a separate terminal:
   ```bash
   kubectl port-forward -n mlpipeline svc/mlpipeline-postgres 5432:5432
   ```
   Then export connection env vars (pulled from the cluster Secret, never printed):
   ```bash
   export POSTGRES_HOST=localhost
   export POSTGRES_PORT=5432
   export POSTGRES_USER=$(kubectl get secret mlpipeline-postgres-credentials -n mlpipeline -o jsonpath='{.data.POSTGRES_USER}' | base64 -d)
   export POSTGRES_PASSWORD=$(kubectl get secret mlpipeline-postgres-credentials -n mlpipeline -o jsonpath='{.data.POSTGRES_PASSWORD}' | base64 -d)
   export POSTGRES_DB=$(kubectl get secret mlpipeline-postgres-credentials -n mlpipeline -o jsonpath='{.data.POSTGRES_DB}' | base64 -d)
   ```
4. Launch Jupyter from the repo root with that shell's env vars intact (e.g. `jupyter notebook notebooks/gdp_etl_preview.ipynb`) so the exported variables above are visible to the kernel.

In [1]:
import os
import sys
from pathlib import Path

import pandas as pd

project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(project_root))

from src.utils.config import load_config
from src.etl.extract import fetch_nipa_data
from src.etl.transform import parse_nipa_response
from src.etl.load import get_connection, ensure_table, upsert_gdp_data

pd.set_option("display.max_columns", None)

## 1. Load config

Same `configs/etl_config.yaml` the DAG's pods read.

In [2]:
config = load_config(str(project_root / "configs" / "etl_config.yaml"))
config

{'bea': {'dataset_name': 'NIPA',
  'table_name': 'T10101',
  'frequency': 'Q',
  'year': 'X'},
 'output': {'raw_data_path': '/data/etl/raw_gdp.json',
  'transformed_data_path': '/data/etl/gdp_tidy.csv'},
 'database': {'table_name': 'gdp'}}

## 2. Extract

Calls `fetch_nipa_data()` -- the same function `extract_gdp_data` runs in the DAG.

In [3]:
bea_cfg = config["bea"]

raw = fetch_nipa_data(
    user_id=os.environ["BEA_API_KEY"],
    table_name=bea_cfg["table_name"],
    frequency=bea_cfg["frequency"],
    year=bea_cfg["year"],
)

print(f"Fetched {len(raw['BEAAPI']['Results']['Data'])} raw records")

Fetched 7900 raw records


## 3. Transform

Calls `parse_nipa_response()` -- the same function `transform_gdp_data` runs in the DAG.

In [4]:
df = parse_nipa_response(raw)
print(df.shape)
df.dtypes

(7900, 5)


period          object
series_code     object
series_name     object
table_name      object
value          float64
dtype: object

In [5]:
df.head(10)

,period,series_code,series_name,table_name,value
0,1947Q2,A191RL,Gross domestic product,T10101,-1.0
1,1947Q3,A191RL,Gross domestic product,T10101,-0.8
2,1947Q4,A191RL,Gross domestic product,T10101,6.4
3,1948Q1,A191RL,Gross domestic product,T10101,6.2
4,1948Q2,A191RL,Gross domestic product,T10101,6.8
5,1948Q3,A191RL,Gross domestic product,T10101,2.3
6,1948Q4,A191RL,Gross domestic product,T10101,0.5
7,1949Q1,A191RL,Gross domestic product,T10101,-5.4
8,1949Q2,A191RL,Gross domestic product,T10101,-1.4
9,1949Q3,A191RL,Gross domestic product,T10101,4.2


In [6]:
print("Period range:", df["period"].min(), "-", df["period"].max())
print("Distinct series:", df["series_code"].nunique())
df["value"].describe()

Period range: 1947Q2 - 2026Q1
Distinct series: 25


count    7900.000000
mean        4.802633
std        15.534653
min       -70.900000
25%        -0.500000
50%         3.700000
75%         8.900000
max       510.600000
Name: value, dtype: float64

## 4. Load (optional)

Calls `ensure_table()` and `upsert_gdp_data()` -- the same functions `load_gdp_data` runs in the DAG. **This writes to the same live `mlpipeline-postgres` instance the production DAG uses**, not a sandbox. The upsert is keyed on `(period, series_code)` and is idempotent (matches BEA's own revision semantics), so re-running it is safe, but it's still a real write -- opt in explicitly.

In [7]:
RUN_LOAD = False  # set True to actually upsert `df` into Postgres

if RUN_LOAD:
    conn = get_connection()
    try:
        ensure_table(conn, config["database"]["table_name"])
        upsert_gdp_data(conn, df, config["database"]["table_name"])
    finally:
        conn.close()
    print(f"Upserted {len(df)} rows into {config['database']['table_name']}")
else:
    print("RUN_LOAD is False -- skipping the write. Set RUN_LOAD = True above to load.")

RUN_LOAD is False -- skipping the write. Set RUN_LOAD = True above to load.


## 5. Preview existing data in Postgres

Reads what's already loaded -- no write required. Needs the port-forward and `POSTGRES_*` env vars from the Prerequisites section.

In [ ]:
table = config["database"]["table_name"]
preview_conn = get_connection()
try:
    schema_df = pd.read_sql(
        "SELECT column_name, data_type, is_nullable "
        "FROM information_schema.columns WHERE table_name = %(t)s "
        "ORDER BY ordinal_position",
        preview_conn,
        params={"t": table},
    )
    sample_df = pd.read_sql(
        f"SELECT * FROM {table} ORDER BY period DESC LIMIT 20", preview_conn
    )
    count_df = pd.read_sql(
        f"SELECT count(*) AS row_count, min(period) AS earliest, max(period) AS latest "
        f"FROM {table}",
        preview_conn,
    )
finally:
    preview_conn.close()

schema_df

In [ ]:
count_df

In [ ]:
sample_df